# RABTECH Academy – Task 5
## Deep Learning / NLP Text Classifier with TensorFlow/Keras
### Project: Movie Review Sentiment Analysis

This notebook classifies movie reviews as **Positive** or **Negative** and displays prediction confidence.

**Requirements covered:** text cleaning and TF-IDF vectorization, a multi-layer neural network, Batch Normalization, Dropout, Early Stopping, training/validation accuracy and loss curves, evaluation on unseen test data, and sample inference.

**Instructions:** Open in Google Colab or Jupyter Notebook and run all cells from top to bottom. Internet access is required to download the IMDB dataset.

## 1. Import libraries

In [ ]:
# TensorFlow is usually pre-installed in Google Colab.
# For local Jupyter, install if needed: %pip install tensorflow scikit-learn matplotlib

import re
import random
import pickle
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
print("TensorFlow version:", tf.__version__)

## 2. Load the IMDB movie-review dataset

In [ ]:
# IMDB provides 25,000 training reviews and 25,000 test reviews.
vocab_size = 10000
(train_sequences, train_labels), (test_sequences, test_labels) = tf.keras.datasets.imdb.load_data(num_words=vocab_size)

word_index = tf.keras.datasets.imdb.get_word_index()
index_to_word = {index + 3: word for word, index in word_index.items()}
index_to_word.update({0: "<PAD>", 1: "<START>", 2: "<UNK>", 3: "<UNUSED>"})

def decode_review(sequence):
    return " ".join(index_to_word.get(int(i), "<UNK>") for i in sequence if i >= 4)

train_texts_all = [decode_review(x) for x in train_sequences]
test_texts_all = [decode_review(x) for x in test_sequences]
y_train_all = np.asarray(train_labels, dtype="int32")
y_test_all = np.asarray(test_labels, dtype="int32")

print("Training reviews:", len(train_texts_all))
print("Test reviews:", len(test_texts_all))
print("Example review:", train_texts_all[0][:400])
print("Example label:", "Positive" if y_train_all[0] else "Negative")

## 3. Select balanced samples for practical training

In [ ]:
# Use a reproducible subset to keep runtime manageable.
train_texts, _, y_train, _ = train_test_split(
    train_texts_all, y_train_all, train_size=10000,
    random_state=SEED, stratify=y_train_all
)
test_texts, _, y_test, _ = train_test_split(
    test_texts_all, y_test_all, train_size=4000,
    random_state=SEED, stratify=y_test_all
)
print("Training samples:", len(train_texts))
print("Test samples:", len(test_texts))
print("Training class counts [negative, positive]:", np.bincount(y_train))

## 4. Clean text and apply TF-IDF vectorization

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"[^a-z\s']", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

train_clean = [clean_text(x) for x in train_texts]
test_clean = [clean_text(x) for x in test_texts]

# Fit the vectorizer on training text only to prevent data leakage.
vectorizer = TfidfVectorizer(
    max_features=3000, ngram_range=(1, 2),
    min_df=2, max_df=0.95, sublinear_tf=True
)
X_train = vectorizer.fit_transform(train_clean).toarray().astype("float32")
X_test = vectorizer.transform(test_clean).toarray().astype("float32")

print("Training feature matrix:", X_train.shape)
print("Test feature matrix:", X_test.shape)
print("TF-IDF feature count:", len(vectorizer.get_feature_names_out()))

## 5. Create training and validation sets

In [ ]:
X_train_part, X_val, y_train_part, y_val = train_test_split(
    X_train, y_train, test_size=0.20,
    random_state=SEED, stratify=y_train
)
print("Training:", X_train_part.shape)
print("Validation:", X_val.shape)
print("Final test:", X_test.shape)

## 6. Build a multi-layer neural network

Dense layers learn patterns in the review features. Batch Normalization helps stabilize training. Dropout reduces overfitting. The sigmoid output produces a probability between 0 and 1.

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(256, activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.40),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.30),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dropout(0.20),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)
model.summary()

## 7. Train with Early Stopping

In [ ]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=3, restore_best_weights=True
)

history = model.fit(
    X_train_part, y_train_part,
    validation_data=(X_val, y_val),
    epochs=15,
    batch_size=64,
    callbacks=[early_stopping],
    verbose=1
)
print("Training completed.")

## 8. Plot training and validation accuracy/loss

In [ ]:
epochs = range(1, len(history.history["loss"]) + 1)

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(epochs, history.history["accuracy"], marker="o", label="Training accuracy")
plt.plot(epochs, history.history["val_accuracy"], marker="o", label="Validation accuracy")
plt.title("Training and Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(epochs, history.history["loss"], marker="o", label="Training loss")
plt.plot(epochs, history.history["val_loss"], marker="o", label="Validation loss")
plt.title("Training and Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## 9. Evaluate on unseen test data

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
test_probabilities = model.predict(X_test, verbose=0).ravel()
test_predictions = (test_probabilities >= 0.5).astype("int32")

print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f} ({test_accuracy * 100:.2f}%)")
print("\nClassification report:")
print(classification_report(
    y_test, test_predictions,
    target_names=["Negative", "Positive"], zero_division=0
))

cm = confusion_matrix(y_test, test_predictions)
ConfusionMatrixDisplay(cm, display_labels=["Negative", "Positive"]).plot(values_format="d")
plt.title("Confusion Matrix – Test Data")
plt.show()

## 10. Predict sentiment for unseen sample reviews

In [ ]:
def predict_sentiment(review):
    cleaned = clean_text(review)
    features = vectorizer.transform([cleaned]).toarray().astype("float32")
    positive_probability = float(model.predict(features, verbose=0)[0][0])

    if positive_probability >= 0.5:
        sentiment = "Positive"
        confidence = positive_probability * 100
    else:
        sentiment = "Negative"
        confidence = (1 - positive_probability) * 100

    print("Review:", review)
    print("Predicted sentiment:", sentiment)
    print(f"Prediction confidence: {confidence:.2f}%")
    return sentiment, confidence

samples = [
    "This movie was wonderful, emotional, and beautifully acted.",
    "The film was boring, poorly written, and a waste of time.",
    "I enjoyed the story and the performances were excellent."
]
for review in samples:
    print("-" * 55)
    predict_sentiment(review)

## 11. Save the trained model and vectorizer

In [ ]:
model.save("sentiment_analysis_model.keras")
with open("tfidf_vectorizer.pkl", "wb") as file:
    pickle.dump(vectorizer, file)
print("Saved sentiment_analysis_model.keras")
print("Saved tfidf_vectorizer.pkl")

## 12. Conclusion and submission checklist

This project uses TensorFlow/Keras to classify movie reviews. Reviews are cleaned and transformed into TF-IDF features. A multi-layer neural network uses Batch Normalization and Dropout, while Early Stopping monitors validation loss. Accuracy/loss curves and a confusion matrix are plotted. The model is evaluated on a separate test set and predicts sentiment with confidence for new reviews.

### Before submitting to RABTECH
1. Run every cell from top to bottom.
2. Confirm that training output, graphs, evaluation metrics and sample predictions appear.
3. Save/download this `.ipynb` notebook.
4. Upload it to a **public GitHub repository**.
5. Open the notebook in GitHub and copy the repository URL.
6. Paste that URL into the Task 5 **Public proof link** field.

**Important:** Use the test accuracy printed by your own run. Do not claim an unverified accuracy.